# AB-SAW 多重結合禁止の対照実験
既存の許可条件18,300試料と、禁止条件18,300試料をサイズ・目標占有率・seedで対応づけます。図の差は「禁止−許可」、誤差棒は対応差の標準誤差です。
既存の許可条件CSVに貫通列がない場合、貫通を比較するには許可条件を現在のコードで再計算し、`baseline_path` をそのCSVに変更します。

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/tetra-peg-network')
baseline_path = root / 'cluster_basics' / 'samples.csv'
study_path = root / 'no_multiple_bonds'
prohibited_path = study_path / 'samples.csv'


In [ ]:
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd

keys = ['L', 'target_phi', 'seed']
metrics = ['p', 'Pmax', 'cluster_weighted_excl_one']
span_metrics = ['spans_x', 'spans_y', 'spans_z']

def compare(baseline_path, prohibited_path, expected_pairs=18_300):
    allowed = pd.read_csv(baseline_path, dtype={'seed': 'string'})
    prohibited = pd.read_csv(prohibited_path, dtype={'seed': 'string'})
    if 'allow_multiple_bonds' in allowed and not allowed['allow_multiple_bonds'].eq(1).all():
        raise ValueError('許可条件のCSVを指定してください')
    if not prohibited[['allow_multiple_bonds', 'H2', 'H3', 'H4']].eq(0).all().all():
        raise ValueError('禁止条件または多重度の検査に失敗しました')
    selected = metrics + [m for m in span_metrics if m in allowed and m in prohibited]
    merged = allowed[keys + ['N', *selected]].merge(
        prohibited[keys + ['N', *selected]], on=keys, how='outer',
        suffixes=('_allow', '_ban'), validate='one_to_one', indicator=True,
    )
    if len(merged) != expected_pairs or not merged['_merge'].eq('both').all():
        raise ValueError('サイズ・目標占有率・seedの対応または試料数が一致しません')
    if not merged['N_allow'].eq(merged['N_ban']).all():
        raise ValueError('許可・禁止間の配置分子数が一致しません')
    if expected_pairs == 18_300:
        counts = merged.groupby(['L', 'target_phi']).size()
        expected = pd.MultiIndex.from_product(
            [[32, 64, 128], [i / 100 for i in range(10, 71)]], names=['L', 'target_phi'])
        if len(counts) != len(expected) or not counts.reindex(expected).eq(100).all():
            raise ValueError('全183条件に100組ずつ必要です')
    for metric in selected:
        merged[f'delta_{metric}'] = merged[f'{metric}_ban'] - merged[f'{metric}_allow']
    group = merged.groupby(['L', 'target_phi'])
    summary = group.agg(**{
        f'{metric}_{label}': (column, method)
        for metric in selected
        for label, column, method in (
            ('allow_mean', f'{metric}_allow', 'mean'), ('ban_mean', f'{metric}_ban', 'mean'),
            ('delta_mean', f'delta_{metric}', 'mean'), ('delta_se', f'delta_{metric}', 'sem'),
            ('valid_pairs', f'delta_{metric}', 'count'),
        )
    }).reset_index()
    return merged, summary, selected


In [ ]:
manifest = json.loads((study_path / 'drive_manifest.json').read_text())
if manifest['samples'] != 18_300:
    raise ValueError('本計算の完了マニフェストが必要です')
if hashlib.sha256(prohibited_path.read_bytes()).hexdigest() != manifest['tables']['samples.csv']:
    raise ValueError('禁止条件の samples.csv のSHA-256が一致しません')
paired, summary, selected = compare(baseline_path, prohibited_path)

columns = 3
fig, axes = plt.subplots((len(selected) + columns - 1) // columns, columns,
                         figsize=(14, 3.8 * ((len(selected) + columns - 1) // columns)),
                         squeeze=False, layout='constrained')
for ax, metric in zip(axes.flat, selected):
    for L, rows in summary.groupby('L'):
        ax.errorbar(rows['target_phi'], rows[f'{metric}_delta_mean'],
                    yerr=rows[f'{metric}_delta_se'], fmt='o', ms=3, capsize=2, label=f'L={L}')
    ax.axhline(0, color='0.55', lw=1)
    ax.set(xlabel='Target occupied fraction', ylabel='Prohibited - allowed', title=metric)
    ax.legend()
for ax in list(axes.flat)[len(selected):]:
    ax.set_visible(False)
plt.show()


In [ ]:
out = study_path / ('comparison_' + pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ'))
out.mkdir()
summary.to_csv(out / 'paired_summary.csv', index=False)
paired.to_csv(out / 'paired_samples.csv', index=False)
fig.savefig(out / 'paired_differences.png', dpi=180, bbox_inches='tight')
print(out)
